# Parallel scan (prefix sum) on the GPU: chapter 11

In session 09 you parallelized a prefix sum on the CPU with `std::async` and three phases (`09_task_parallelism/example_54.cpp`, `exercise_9_2.cpp`). This folder does the same thing on a GPU. There are thousands of threads instead of eight, so the question changes. It is no longer "how do I split the array into a few chunks" but "how do 256 threads that share a small scratchpad scan their piece in a few steps, and how do thousands of such pieces agree on their offsets".

This notebook runs the **complete programs**: every kernel of the chapter on large inputs, compared side by side with timings, and the **bike-sharing examples**, which apply the same pattern to another domain to give you ideas for your project. Do the step-by-step notebook (`colab_1_step_by_step.ipynb`) first. Timings are only meaningful on the GPU.

The full theory, the exercises and their solutions are in `ec2/09_scan/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_group_by_station.cu`

counting-sort bucketing of trips by start station: histogram → exclusive scan → scatter (atomic and stable)

This is a bike-sharing example: the README section *Ideas for your project* explains it and lists where the same pattern may appear in your project.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile bikeshare_group_by_station.cu
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

#include "cuda_compat.h"

constexpr int TRIPS_PER_BLOCK = 256;
constexpr int BK_THREADS = 256;
constexpr int BK_SECTION = 2 * BK_THREADS;
constexpr int MAX_STATIONS = BK_SECTION;

constexpr int OFFSET_THREADS = 256;
constexpr int STATION_THREADS = 64;
constexpr int MAX_HISTOGRAM_BLOCKS = BY_BACKEND(1024, 16);

#ifndef CPU_TRIPS
#define CPU_TRIPS 60000
#endif
constexpr int DEFAULT_TRIPS = BY_BACKEND(4000000, CPU_TRIPS);
constexpr int STATIONS = 48;
constexpr int TIMED_RUNS = BY_BACKEND(5, 1);
constexpr int WARMUP_RUNS = BY_BACKEND(1, 0);
constexpr uint64_t TRIP_LOG_SEED = 17;

constexpr int POPULARITY_SCALE = 1200;
constexpr int QUIET_STATION_WEIGHT = 15;

constexpr int MS_PER_SECOND = 1000;
constexpr int MS_PER_MINUTE = 60 * MS_PER_SECOND;
constexpr int MS_PER_HOUR = 60 * MS_PER_MINUTE;
constexpr int HOURS_PER_DAY = 24;
constexpr int HOUR_WEIGHTS[HOURS_PER_DAY] = {1, 1, 1, 1, 1, 2,  5,  12, 20, 12, 6, 5,
                                             7, 7, 5, 5, 7, 13, 20, 13, 7,  4,  3, 2};

constexpr int BUSIEST_STATIONS_SHOWN = 5;
constexpr int FIRST_TRIPS_SHOWN = 3;

static_assert(STATIONS <= MAX_STATIONS,
              "the starts of all stations are scanned by one Brent-Kung block");

__device__ void brent_kung_in_shared(int* XY) {
  const int t = threadIdx.x;
  for (int stride = 1; stride <= BK_THREADS; stride *= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index < BK_SECTION) {
      XY[index] += XY[index - stride];
    }
  }

  for (int stride = BK_SECTION / 4; stride > 0; stride /= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index + stride < BK_SECTION) {
      XY[index + stride] += XY[index];
    }
  }
  __syncthreads();
}

__global__ void brent_kung_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  const int base = blockIdx.x * BK_SECTION;
  XY[t] = (base + t < n) ? X[base + t] : 0;
  XY[t + BK_THREADS] = (base + t + BK_THREADS < n) ? X[base + t + BK_THREADS] : 0;

  brent_kung_in_shared(XY);

  if (base + t < n) {
    Y[base + t] = XY[t];
  }
  if (base + t + BK_THREADS < n) {
    Y[base + t + BK_THREADS] = XY[t + BK_THREADS];
  }
  if (t == 0) {
    sums[blockIdx.x] = XY[BK_SECTION - 1];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n, int section) {
  if (blockIdx.x > 0) {
    const int add = scannedSums[blockIdx.x - 1];
    for (int j = threadIdx.x; j < section; j += blockDim.x) {
      const long long idx = (long long)blockIdx.x * section + j;
      if (idx < n) {
        Y[idx] += add;
      }
    }
  }
}

__host__ __device__ inline int exclusive_from_inclusive(const int* inclusive, int i) {
  return (i > 0) ? inclusive[i - 1] : 0;
}

__host__ __device__ inline int station_major_cell(int station, int block, int blocks) {
  return station * blocks + block;
}

__device__ void clear_histogram(int* histogram, int stations) {
  for (int s = threadIdx.x; s < stations; s += blockDim.x) {
    histogram[s] = 0;
  }
}

__global__ void count_per_station(const int* startStation, int* counts, int n, int stations) {
  __shared__ int histogram[MAX_STATIONS];
  clear_histogram(histogram, stations);
  __syncthreads();

  const int stride = blockDim.x * gridDim.x;
  for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
    atomicAdd(&histogram[startStation[i]], 1);
  }
  __syncthreads();

  for (int s = threadIdx.x; s < stations; s += blockDim.x) {
    if (histogram[s] > 0) {
      atomicAdd(&counts[s], histogram[s]);
    }
  }
}

__global__ void bucket_starts(const int* counts, int* start, int stations) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  XY[t] = (t < stations) ? counts[t] : 0;
  XY[t + BK_THREADS] = (t + BK_THREADS < stations) ? counts[t + BK_THREADS] : 0;

  brent_kung_in_shared(XY);

  if (t == 0) {
    start[0] = 0;
  }
  if (t < stations) {
    start[t + 1] = XY[t];
  }
  if (t + BK_THREADS < stations) {
    start[t + BK_THREADS + 1] = XY[t + BK_THREADS];
  }
}

__global__ void scatter_by_cursor(const int* startStation, int* cursor, int* grouped, int n) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    const int slot = atomicAdd(&cursor[startStation[i]], 1);
    grouped[slot] = i;
  }
}

__global__ void block_histograms(const int* startStation, int* counts, int n, int stations) {
  __shared__ int histogram[MAX_STATIONS];
  clear_histogram(histogram, stations);
  __syncthreads();

  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    atomicAdd(&histogram[startStation[i]], 1);
  }
  __syncthreads();

  for (int s = threadIdx.x; s < stations; s += blockDim.x) {
    counts[station_major_cell(s, blockIdx.x, gridDim.x)] = histogram[s];
  }
}

__device__ int earlier_trips_from_same_station_in_block(const int* stationOfThread, int t) {
  int rank = 0;
  for (int j = 0; j < t; ++j) {
    if (stationOfThread[j] == stationOfThread[t]) {
      ++rank;
    }
  }
  return rank;
}

__global__ void stable_scatter(const int* startStation, const int* scanned, int* grouped, int n) {
  __shared__ int stationOfThread[TRIPS_PER_BLOCK];
  const int t = threadIdx.x;
  const int i = blockIdx.x * TRIPS_PER_BLOCK + t;
  const int s = (i < n) ? startStation[i] : -1;
  stationOfThread[t] = s;
  __syncthreads();

  if (i < n) {
    const int rank = earlier_trips_from_same_station_in_block(stationOfThread, t);
    const int cell = station_major_cell(s, blockIdx.x, gridDim.x);
    grouped[exclusive_from_inclusive(scanned, cell) + rank] = i;
  }
}

__global__ void starts_from_matrix(const int* scanned, int* start, int stations, int blocks) {
  const int s = blockIdx.x * blockDim.x + threadIdx.x;
  if (s < stations) {
    start[s] = exclusive_from_inclusive(scanned, station_major_cell(s, 0, blocks));
  }
  if (s == 0) {
    start[stations] = scanned[station_major_cell(stations, 0, blocks) - 1];
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int bound) {
    return static_cast<int>(next() % static_cast<uint64_t>(bound));
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

class WeightedPick {
 public:
  explicit WeightedPick(const std::vector<int>& weights) : cumulative_(weights.size()) {
    int total = 0;
    for (size_t k = 0; k < weights.size(); ++k) {
      total += weights[k];
      cumulative_[k] = total;
    }
  }

  int pick(SplitMix64& rng) const {
    const int ticket = rng.below(cumulative_.back());
    const auto chosen = std::upper_bound(cumulative_.begin(), cumulative_.end(), ticket);
    return static_cast<int>(chosen - cumulative_.begin());
  }

 private:
  std::vector<int> cumulative_;
};

struct TripLog {
  std::vector<int> startStation;
  std::vector<int> startMs;
};

struct Grouping {
  std::vector<int> start;
  std::vector<int> grouped;
};

struct DeviceBuffers {
  int* startStation;
  int* counts;
  int* start;
  int* cursor;
  int* grouped;
  int* matrix;
  int* scannedMatrix;
  int* scratch;
};

int blocksFor(int n, int blockSize) {
  return static_cast<int>(cdiv(n, blockSize));
}

int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(int)));
  return pointer;
}

int scratchIntsForHierarchicalScan(int n) {
  const int blocks = blocksFor(n, BK_SECTION);
  if (blocks == 1) {
    return 1;
  }
  return 2 * blocks + scratchIntsForHierarchicalScan(blocks);
}

void scanInclusive(const int* d_in, int* d_out, int n, int* scratch) {
  const int blocks = blocksFor(n, BK_SECTION);
  int* blockSums = scratch;
  launch(brent_kung_scan, dim3(blocks), dim3(BK_THREADS), d_in, d_out, blockSums, n);

  if (blocks > 1) {
    int* scannedBlockSums = scratch + blocks;
    int* scratchForNextLevel = scratch + 2 * blocks;
    scanInclusive(blockSums, scannedBlockSums, blocks, scratchForNextLevel);
    launch(add_block_offsets,
           dim3(blocks),
           dim3(OFFSET_THREADS),
           d_out,
           scannedBlockSums,
           n,
           BK_SECTION);
  }
}

std::vector<int> stationPopularity() {
  std::vector<int> weights(STATIONS);
  for (int s = 0; s < STATIONS; ++s) {
    weights[s] = POPULARITY_SCALE / (s + 1) + QUIET_STATION_WEIGHT;
  }
  return weights;
}

std::vector<int> sortedStartTimes(SplitMix64& rng, int trips) {
  const WeightedPick pickHour(std::vector<int>(std::begin(HOUR_WEIGHTS), std::end(HOUR_WEIGHTS)));
  std::vector<int> startMs(trips);
  for (int i = 0; i < trips; ++i) {
    startMs[i] = pickHour.pick(rng) * MS_PER_HOUR + rng.below(MS_PER_HOUR);
  }
  std::sort(startMs.begin(), startMs.end());
  return startMs;
}

TripLog makeTripLog(int trips) {
  SplitMix64 rng(TRIP_LOG_SEED);
  const WeightedPick pickStation(stationPopularity());

  TripLog log;
  log.startMs = sortedStartTimes(rng, trips);
  log.startStation.resize(trips);
  for (int i = 0; i < trips; ++i) {
    log.startStation[i] = pickStation.pick(rng);
  }
  return log;
}

Grouping cpuStableCountingSort(const std::vector<int>& startStation) {
  const int trips = static_cast<int>(startStation.size());
  std::vector<int> counts(STATIONS, 0);
  for (int s : startStation) {
    ++counts[s];
  }

  Grouping grouping;
  grouping.start.assign(STATIONS + 1, 0);
  for (int s = 0; s < STATIONS; ++s) {
    grouping.start[s + 1] = grouping.start[s] + counts[s];
  }

  std::vector<int> cursor(grouping.start.begin(), grouping.start.end() - 1);
  grouping.grouped.resize(trips);
  for (int i = 0; i < trips; ++i) {
    grouping.grouped[cursor[startStation[i]]++] = i;
  }
  return grouping;
}

DeviceBuffers allocateDeviceBuffers(int trips) {
  const int matrixCells = STATIONS * blocksFor(trips, TRIPS_PER_BLOCK);
  DeviceBuffers buffers;
  buffers.startStation = allocateDeviceInts(trips);
  buffers.counts = allocateDeviceInts(STATIONS);
  buffers.start = allocateDeviceInts(STATIONS + 1);
  buffers.cursor = allocateDeviceInts(STATIONS + 1);
  buffers.grouped = allocateDeviceInts(trips);
  buffers.matrix = allocateDeviceInts(matrixCells);
  buffers.scannedMatrix = allocateDeviceInts(matrixCells);
  buffers.scratch = allocateDeviceInts(scratchIntsForHierarchicalScan(matrixCells));
  return buffers;
}

void freeDeviceBuffers(const DeviceBuffers& buffers) {
  CHECK(cudaFree(buffers.startStation));
  CHECK(cudaFree(buffers.counts));
  CHECK(cudaFree(buffers.start));
  CHECK(cudaFree(buffers.cursor));
  CHECK(cudaFree(buffers.grouped));
  CHECK(cudaFree(buffers.matrix));
  CHECK(cudaFree(buffers.scannedMatrix));
  CHECK(cudaFree(buffers.scratch));
}

void groupWithAtomicCursor(const DeviceBuffers& buffers, int trips) {
  const int tripBlocks = blocksFor(trips, TRIPS_PER_BLOCK);
  const int histogramBlocks = imin(tripBlocks, MAX_HISTOGRAM_BLOCKS);
  const size_t startBytes = (STATIONS + 1) * sizeof(int);

  CHECK(cudaMemset(buffers.counts, 0, STATIONS * sizeof(int)));
  launch(count_per_station,
         dim3(histogramBlocks),
         dim3(TRIPS_PER_BLOCK),
         buffers.startStation,
         buffers.counts,
         trips,
         STATIONS);
  launch(bucket_starts, dim3(1), dim3(BK_THREADS), buffers.counts, buffers.start, STATIONS);
  CHECK(cudaMemcpy(buffers.cursor, buffers.start, startBytes, cudaMemcpyDeviceToDevice));
  launch(scatter_by_cursor,
         dim3(tripBlocks),
         dim3(TRIPS_PER_BLOCK),
         buffers.startStation,
         buffers.cursor,
         buffers.grouped,
         trips);
}

void groupStably(const DeviceBuffers& buffers, int trips) {
  const int tripBlocks = blocksFor(trips, TRIPS_PER_BLOCK);
  const int matrixCells = STATIONS * tripBlocks;

  launch(block_histograms,
         dim3(tripBlocks),
         dim3(TRIPS_PER_BLOCK),
         buffers.startStation,
         buffers.matrix,
         trips,
         STATIONS);
  scanInclusive(buffers.matrix, buffers.scannedMatrix, matrixCells, buffers.scratch);
  launch(stable_scatter,
         dim3(tripBlocks),
         dim3(TRIPS_PER_BLOCK),
         buffers.startStation,
         buffers.scannedMatrix,
         buffers.grouped,
         trips);
  launch(starts_from_matrix,
         dim3(blocksFor(STATIONS, STATION_THREADS)),
         dim3(STATION_THREADS),
         buffers.scannedMatrix,
         buffers.start,
         STATIONS,
         tripBlocks);
}

void fillWithMinusOne(int* d_values, int count) {
  CHECK(cudaMemset(d_values, 0xff, count * sizeof(int)));
}

template <class Work>
double averageMs(int runs, Work work) {
  Timer timer;
  for (int run = 0; run < runs; ++run) {
    work();
  }
  return timer.ms() / runs;
}

template <class Work>
double measureGrouping(Work work, const DeviceBuffers& buffers, int trips) {
  for (int warmup = 0; warmup < WARMUP_RUNS; ++warmup) {
    work();
  }
  fillWithMinusOne(buffers.grouped, trips);
  CHECK(cudaDeviceSynchronize());
  const double ms = averageMs(TIMED_RUNS, work);
  CHECK(cudaGetLastError());
  return ms;
}

Grouping readGrouping(const DeviceBuffers& buffers, int trips) {
  Grouping grouping;
  grouping.start.resize(STATIONS + 1);
  grouping.grouped.resize(trips);
  CHECK(cudaMemcpy(
      grouping.start.data(), buffers.start, (STATIONS + 1) * sizeof(int), cudaMemcpyDeviceToHost));
  CHECK(cudaMemcpy(
      grouping.grouped.data(), buffers.grouped, trips * sizeof(int), cudaMemcpyDeviceToHost));
  return grouping;
}

std::vector<int> sliceOf(const Grouping& grouping, int station) {
  return std::vector<int>(grouping.grouped.begin() + grouping.start[station],
                          grouping.grouped.begin() + grouping.start[station + 1]);
}

struct SliceCheck {
  bool sameTripsPerSlice;
  int slicesByStartTime;
};

SliceCheck compareSlicesAsSets(const Grouping& actual, const Grouping& expected) {
  SliceCheck check = {actual.start == expected.start, 0};
  for (int s = 0; s < STATIONS && check.sameTripsPerSlice; ++s) {
    std::vector<int> slice = sliceOf(actual, s);
    if (std::is_sorted(slice.begin(), slice.end())) {
      ++check.slicesByStartTime;
    }
    std::sort(slice.begin(), slice.end());
    check.sameTripsPerSlice = (slice == sliceOf(expected, s));
  }
  return check;
}

const char* verdict(bool pass) {
  return pass ? "PASS" : "FAIL";
}

void printClock(int ms) {
  std::printf("%02d:%02d:%02d", ms / MS_PER_HOUR, ms / MS_PER_MINUTE % 60, ms / MS_PER_SECOND % 60);
}

int sliceLength(const Grouping& grouping, int station) {
  return grouping.start[station + 1] - grouping.start[station];
}

std::vector<int> stationsByTripCount(const Grouping& grouping) {
  std::vector<int> stations(STATIONS);
  for (int s = 0; s < STATIONS; ++s) {
    stations[s] = s;
  }
  std::stable_sort(stations.begin(), stations.end(), [&](int a, int b) {
    return sliceLength(grouping, a) > sliceLength(grouping, b);
  });
  return stations;
}

void printBusiestStations(const Grouping& expected, const Grouping& grouping, const TripLog& log) {
  const std::vector<int> busiest = stationsByTripCount(expected);
  std::printf("\nBusiest start stations (slice of the grouped array from version B):\n");
  for (int k = 0; k < BUSIEST_STATIONS_SHOWN; ++k) {
    const int s = busiest[k];
    const int first = grouping.start[s];
    const int shownEnd = imin(first + FIRST_TRIPS_SHOWN, grouping.start[s + 1]);
    std::printf("  s%-3d %7d trips  start %8d  first:", s, sliceLength(grouping, s), first);
    for (int j = first; j < shownEnd; ++j) {
      const int trip = grouping.grouped[j];
      std::printf("  t%d @ ", trip);
      printClock(log.startMs[trip]);
    }
    std::printf("\n");
  }
}

void printEmulationNote() {
#ifndef __CUDACC__
  std::printf("\nNote: CPU emulation times measure thread barriers, not a GPU.\n");
#endif
}

int tripCount(int argc, char** argv) {
  return (argc > 1) ? std::atoi(argv[1]) : DEFAULT_TRIPS;
}

void printAtomicRow(double cpuMs, double atomicMs, const SliceCheck& check) {
  std::printf("  %-40s %10.3f %7.2fx  %s (same trips per slice; %d of %d slices by start time)\n",
              "A  histogram + scan + atomic cursor",
              atomicMs,
              cpuMs / atomicMs,
              verdict(check.sameTripsPerSlice),
              check.slicesByStartTime,
              STATIONS);
}

void printStableRow(double cpuMs, double stableMs, bool pass) {
  std::printf("  %-40s %10.3f %7.2fx  %s (identical to the CPU, every slice by start time)\n",
              "B  block histograms + scan + stable rank",
              stableMs,
              cpuMs / stableMs,
              verdict(pass));
}

int main(int argc, char** argv) {
  const int trips = tripCount(argc, argv);
  if (trips < 1) {
    std::fprintf(stderr, "need at least one trip\n");
    return 1;
  }
  std::printf(
      "Group trips by start station (counting sort = histogram + exclusive scan + scatter)\n");
  std::printf("backend: %s, %d trips, %d stations\n\n", CUDA_BACKEND, trips, STATIONS);

  const TripLog log = makeTripLog(trips);
  Grouping expected;
  const double cpuMs = averageMs(TIMED_RUNS, [&] {
    expected = cpuStableCountingSort(log.startStation);
  });

  const DeviceBuffers buffers = allocateDeviceBuffers(trips);
  CHECK(cudaMemcpy(
      buffers.startStation, log.startStation.data(), trips * sizeof(int), cudaMemcpyHostToDevice));

  std::printf("  %-40s %10s %8s  %s\n", "version", "time [ms]", "vs CPU", "check");
  std::printf(
      "  %-40s %10.3f %8s  %s\n", "CPU stable counting sort (reference)", cpuMs, "1.00x", "-");

  const double atomicMs = measureGrouping(
      [&] {
        groupWithAtomicCursor(buffers, trips);
      },
      buffers,
      trips);
  const SliceCheck atomicCheck = compareSlicesAsSets(readGrouping(buffers, trips), expected);
  printAtomicRow(cpuMs, atomicMs, atomicCheck);

  const double stableMs = measureGrouping(
      [&] {
        groupStably(buffers, trips);
      },
      buffers,
      trips);
  const Grouping stable = readGrouping(buffers, trips);
  const bool stablePass = (stable.start == expected.start) && (stable.grouped == expected.grouped);
  printStableRow(cpuMs, stableMs, stablePass);

  printBusiestStations(expected, stable, log);
  printEmulationNote();

  const bool allPass = atomicCheck.sameTripsPerSlice && stablePass;
  std::printf("\n%s\n", allPass ? "ALL PASS" : "SOME CHECKS FAILED");

  freeDeviceBuffers(buffers);
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_group_by_station.cu -o bikeshare_group_by_station && ./bikeshare_group_by_station

## `bikeshare_rentals_in_progress.cu`

+1 undock / −1 dock events → inclusive scan → peak of bikes out and the hourly curve

This is a bike-sharing example: the README section *Ideas for your project* explains it and lists where the same pattern may appear in your project.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile bikeshare_rentals_in_progress.cu
#include <algorithm>
#include <climits>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

#include "cuda_compat.h"

constexpr int CO_THREADS = 256;
constexpr int CO_COARSEN = 8;
constexpr int CO_SECTION = CO_THREADS * CO_COARSEN;

constexpr int OFFSET_THREADS = 256;
constexpr int REDUCTION_THREADS = 256;
constexpr int MAX_REDUCTION_BLOCKS = BY_BACKEND(1024, 8);

#ifndef CPU_TRIPS
#define CPU_TRIPS 40000
#endif
constexpr int DEFAULT_TRIPS = BY_BACKEND(4000000, CPU_TRIPS);
constexpr int TIMED_RUNS = BY_BACKEND(5, 1);
constexpr int WARMUP_RUNS = BY_BACKEND(1, 0);
constexpr uint64_t TRIP_LOG_SEED = 42;

constexpr int MS_PER_SECOND = 1000;
constexpr int MS_PER_MINUTE = 60 * MS_PER_SECOND;
constexpr int MS_PER_HOUR = 60 * MS_PER_MINUTE;
constexpr int HOURS_PER_DAY = 24;
constexpr int DAY_MS = HOURS_PER_DAY * MS_PER_HOUR;
constexpr int END_OF_RUN_MS = DAY_MS + 3 * MS_PER_HOUR;

constexpr int PERCENT = 100;
constexpr int FAULTY_UNDOCK_PERCENT = 3;
constexpr int LEISURE_RIDE_PERCENT = 12;
constexpr int MAX_FAULTY_CHECK_SECONDS = 90;
constexpr int MIN_COMMUTE_MINUTES = 4;
constexpr int MAX_COMMUTE_MINUTES = 35;
constexpr int MIN_LEISURE_MINUTES = 30;
constexpr int MAX_LEISURE_MINUTES = 150;

constexpr int HOUR_WEIGHTS[HOURS_PER_DAY] = {1, 1, 1, 1, 1, 2,  5,  12, 20, 12, 6, 5,
                                             7, 7, 5, 5, 7, 13, 20, 13, 7,  4,  3, 2};

constexpr int BIKE_LEAVES_DOCK = +1;
constexpr int BIKE_RETURNS_TO_DOCK = -1;
constexpr int BAR_LENGTH_AT_PEAK = 50;

__device__ void load_section_coalesced(const int* X, int n, int base, int* XY) {
  for (int k = 0; k < CO_COARSEN; ++k) {
    const int j = k * CO_THREADS + threadIdx.x;
    XY[j] = (base + j < n) ? X[base + j] : 0;
  }
}

__device__ void store_section_coalesced(const int* XY, int n, int base, int* Y) {
  for (int k = 0; k < CO_COARSEN; ++k) {
    const int j = k * CO_THREADS + threadIdx.x;
    if (base + j < n) {
      Y[base + j] = XY[j];
    }
  }
}

__device__ void phase1_scan_own_subsection(int* XY, int first) {
  for (int k = 1; k < CO_COARSEN; ++k) {
    XY[first + k] += XY[first + k - 1];
  }
}

__device__ void phase2_kogge_stone_over_subsection_totals(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < CO_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ void phase3_add_total_of_subsections_to_the_left(int* XY, const int* part, int first) {
  const int t = threadIdx.x;
  if (t > 0) {
    const int add = part[t - 1];
    for (int k = 0; k < CO_COARSEN; ++k) {
      XY[first + k] += add;
    }
  }
}

__global__ void coarsened_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[CO_SECTION];
  __shared__ int part[CO_THREADS];
  const int t = threadIdx.x;
  const int base = blockIdx.x * CO_SECTION;
  const int first = t * CO_COARSEN;

  load_section_coalesced(X, n, base, XY);
  __syncthreads();

  phase1_scan_own_subsection(XY, first);
  part[t] = XY[first + CO_COARSEN - 1];

  phase2_kogge_stone_over_subsection_totals(part);
  __syncthreads();

  phase3_add_total_of_subsections_to_the_left(XY, part, first);
  __syncthreads();

  store_section_coalesced(XY, n, base, Y);
  if (t == 0) {
    sums[blockIdx.x] = part[CO_THREADS - 1];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n, int section) {
  if (blockIdx.x > 0) {
    const int add = scannedSums[blockIdx.x - 1];
    for (int j = threadIdx.x; j < section; j += blockDim.x) {
      const long long idx = (long long)blockIdx.x * section + j;
      if (idx < n) {
        Y[idx] += add;
      }
    }
  }
}

__global__ void max_value(const int* v, int* result, int n) {
  __shared__ int blockMax;
  if (threadIdx.x == 0) {
    blockMax = INT_MIN;
  }
  __syncthreads();

  int threadMax = INT_MIN;
  const int stride = blockDim.x * gridDim.x;
  for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
    threadMax = imax(threadMax, v[i]);
  }
  atomicMax(&blockMax, threadMax);
  __syncthreads();

  if (threadIdx.x == 0) {
    atomicMax(result, blockMax);
  }
}

__global__ void first_index_of(const int* v, int value, int* result, int n) {
  const int stride = blockDim.x * gridDim.x;
  for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
    if (v[i] == value) {
      atomicMin(result, i);
    }
  }
}

enum class DockEventKind { Undock, Dock };

struct DockEvent {
  int timeMs;
  DockEventKind kind;
  int delta;
};

bool happensBefore(const DockEvent& a, const DockEvent& b) {
  if (a.timeMs != b.timeMs) {
    return a.timeMs < b.timeMs;
  }
  return a.kind < b.kind;
}

struct EventStream {
  std::vector<int> deltas;
  std::vector<int> timesMs;
  int faultyUndocks;
};

struct PeakLoad {
  int rentalsOut;
  int firstEvent;
};

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int bound) {
    return static_cast<int>(next() % static_cast<uint64_t>(bound));
  }

  int between(int lowest, int highest) {
    return lowest + below(highest - lowest + 1);
  }

  bool percentChance(int percent) {
    return below(PERCENT) < percent;
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

int blocksFor(int n, int blockSize) {
  return static_cast<int>(cdiv(n, blockSize));
}

int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(int)));
  return pointer;
}

void copyIntToDevice(int* destination, int value) {
  CHECK(cudaMemcpy(destination, &value, sizeof(int), cudaMemcpyHostToDevice));
}

int copyIntToHost(const int* source) {
  int value = 0;
  CHECK(cudaMemcpy(&value, source, sizeof(int), cudaMemcpyDeviceToHost));
  return value;
}

int scratchIntsForHierarchicalScan(int n) {
  const int blocks = blocksFor(n, CO_SECTION);
  if (blocks == 1) {
    return 1;
  }
  return 2 * blocks + scratchIntsForHierarchicalScan(blocks);
}

void scanInclusive(const int* d_in, int* d_out, int n, int* scratch) {
  const int blocks = blocksFor(n, CO_SECTION);
  int* blockSums = scratch;
  launch(coarsened_scan, dim3(blocks), dim3(CO_THREADS), d_in, d_out, blockSums, n);

  if (blocks > 1) {
    int* scannedBlockSums = scratch + blocks;
    int* scratchForNextLevel = scratch + 2 * blocks;
    scanInclusive(blockSums, scannedBlockSums, blocks, scratchForNextLevel);
    launch(add_block_offsets,
           dim3(blocks),
           dim3(OFFSET_THREADS),
           d_out,
           scannedBlockSums,
           n,
           CO_SECTION);
  }
}

int minutesToMs(int minutes) {
  return minutes * MS_PER_MINUTE;
}

class SyntheticDay {
 public:
  SyntheticDay() : rng_(TRIP_LOG_SEED), hourTotal_(0) {
    for (int hour = 0; hour < HOURS_PER_DAY; ++hour) {
      hourTotal_ += HOUR_WEIGHTS[hour];
      cumulativeHourWeight_[hour] = hourTotal_;
    }
  }

  void addTrip(std::vector<DockEvent>& events, int& faultyUndocks) {
    const int undockMs = pickHour() * MS_PER_HOUR + rng_.below(MS_PER_HOUR);
    int dockMs = 0;
    if (rng_.percentChance(FAULTY_UNDOCK_PERCENT)) {
      dockMs = undockMs + rng_.below(MAX_FAULTY_CHECK_SECONDS * MS_PER_SECOND);
      ++faultyUndocks;
    } else {
      dockMs = undockMs + rideMs();
    }

    events.push_back({undockMs, DockEventKind::Undock, BIKE_LEAVES_DOCK});
    events.push_back({imin(dockMs, END_OF_RUN_MS), DockEventKind::Dock, BIKE_RETURNS_TO_DOCK});
  }

 private:
  int pickHour() {
    const int ticket = rng_.below(hourTotal_);
    int hour = 0;
    while (cumulativeHourWeight_[hour] <= ticket) {
      ++hour;
    }
    return hour;
  }

  int rideMs() {
    if (rng_.percentChance(LEISURE_RIDE_PERCENT)) {
      return rng_.between(minutesToMs(MIN_LEISURE_MINUTES), minutesToMs(MAX_LEISURE_MINUTES));
    }
    return rng_.between(minutesToMs(MIN_COMMUTE_MINUTES), minutesToMs(MAX_COMMUTE_MINUTES));
  }

  SplitMix64 rng_;
  int hourTotal_;
  int cumulativeHourWeight_[HOURS_PER_DAY];
};

EventStream makeEventStream(int trips) {
  SyntheticDay day;
  std::vector<DockEvent> events;
  events.reserve(2 * trips);
  int faultyUndocks = 0;
  for (int trip = 0; trip < trips; ++trip) {
    day.addTrip(events, faultyUndocks);
  }
  std::sort(events.begin(), events.end(), happensBefore);

  EventStream stream;
  stream.faultyUndocks = faultyUndocks;
  for (const DockEvent& event : events) {
    stream.deltas.push_back(event.delta);
    stream.timesMs.push_back(event.timeMs);
  }
  return stream;
}

PeakLoad cpuRunningSumAndPeak(const std::vector<int>& deltas, std::vector<int>& out) {
  PeakLoad peak = {INT_MIN, -1};
  int run = 0;
  for (size_t i = 0; i < deltas.size(); ++i) {
    run += deltas[i];
    out[i] = run;
    if (run > peak.rentalsOut) {
      peak.rentalsOut = run;
      peak.firstEvent = static_cast<int>(i);
    }
  }
  return peak;
}

struct DeviceBuffers {
  int* deltas;
  int* out;
  int* scratch;
  int* peak;
  int* peakEvent;
};

DeviceBuffers allocateDeviceBuffers(int events) {
  DeviceBuffers buffers;
  buffers.deltas = allocateDeviceInts(events);
  buffers.out = allocateDeviceInts(events);
  buffers.scratch = allocateDeviceInts(scratchIntsForHierarchicalScan(events));
  buffers.peak = allocateDeviceInts(1);
  buffers.peakEvent = allocateDeviceInts(1);
  return buffers;
}

void freeDeviceBuffers(const DeviceBuffers& buffers) {
  CHECK(cudaFree(buffers.deltas));
  CHECK(cudaFree(buffers.out));
  CHECK(cudaFree(buffers.scratch));
  CHECK(cudaFree(buffers.peak));
  CHECK(cudaFree(buffers.peakEvent));
}

PeakLoad gpuRentalsOutAndPeak(const DeviceBuffers& buffers, int events) {
  const int blocks = imin(blocksFor(events, REDUCTION_THREADS), MAX_REDUCTION_BLOCKS);
  scanInclusive(buffers.deltas, buffers.out, events, buffers.scratch);

  copyIntToDevice(buffers.peak, INT_MIN);
  copyIntToDevice(buffers.peakEvent, INT_MAX);
  launch(max_value, dim3(blocks), dim3(REDUCTION_THREADS), buffers.out, buffers.peak, events);

  PeakLoad peak;
  peak.rentalsOut = copyIntToHost(buffers.peak);
  launch(first_index_of,
         dim3(blocks),
         dim3(REDUCTION_THREADS),
         buffers.out,
         peak.rentalsOut,
         buffers.peakEvent,
         events);
  peak.firstEvent = copyIntToHost(buffers.peakEvent);
  return peak;
}

template <class Work>
double averageMs(int runs, Work work) {
  Timer timer;
  for (int run = 0; run < runs; ++run) {
    work();
  }
  return timer.ms() / runs;
}

const char* verdict(bool pass) {
  return pass ? "PASS" : "FAIL";
}

void printClock(int ms) {
  std::printf("%02d:%02d", ms / MS_PER_HOUR, ms / MS_PER_MINUTE % 60);
}

int rentalsOutAtTime(int timeMs, const std::vector<int>& timesMs, const std::vector<int>& out) {
  const auto firstEventAtOrAfter = std::lower_bound(timesMs.begin(), timesMs.end(), timeMs);
  const int lastEventBefore = static_cast<int>(firstEventAtOrAfter - timesMs.begin()) - 1;
  return (lastEventBefore >= 0) ? out[lastEventBefore] : 0;
}

void printHourlyCurve(const EventStream& stream, const std::vector<int>& out, int peakRentals) {
  const int rentalsPerMark = imax(1, peakRentals / BAR_LENGTH_AT_PEAK);
  std::printf("\nBikes out on a rental at each full hour (each # = %d bikes):\n", rentalsPerMark);
  for (int hour = 1; hour <= HOURS_PER_DAY; ++hour) {
    const int rentalsOut = rentalsOutAtTime(hour * MS_PER_HOUR, stream.timesMs, out);
    std::printf("  %02d:00 %6d ", hour, rentalsOut);
    for (int mark = 0; mark < rentalsOut / rentalsPerMark; ++mark) {
      std::printf("#");
    }
    std::printf("\n");
  }
}

void printEmulationNote() {
#ifndef __CUDACC__
  std::printf("\nNote: CPU emulation times measure thread barriers, not a GPU.\n");
#endif
}

int tripCount(int argc, char** argv) {
  return (argc > 1) ? std::atoi(argv[1]) : DEFAULT_TRIPS;
}

void printPeak(int trips, const EventStream& stream, const PeakLoad& peak) {
  std::printf("\n%d trips (%d faulty undocks returned within %d s). Peak: %d bikes out at once, ",
              trips,
              stream.faultyUndocks,
              MAX_FAULTY_CHECK_SECONDS,
              peak.rentalsOut);
  std::printf("first reached at ");
  printClock(stream.timesMs[peak.firstEvent]);
  std::printf(" (event %d).\n", peak.firstEvent);
}

int main(int argc, char** argv) {
  const int trips = tripCount(argc, argv);
  if (trips < 1) {
    std::fprintf(stderr, "need at least one trip\n");
    return 1;
  }
  const int events = 2 * trips;
  std::printf("Bikes out on a rental over one day (+1 undock, -1 dock, inclusive scan)\n");
  std::printf("backend: %s, %d trips -> %d events\n\n", CUDA_BACKEND, trips, events);

  const EventStream stream = makeEventStream(trips);

  std::vector<int> expectedOut(events);
  PeakLoad expectedPeak = {};
  const double cpuMs = averageMs(TIMED_RUNS, [&] {
    expectedPeak = cpuRunningSumAndPeak(stream.deltas, expectedOut);
  });

  const DeviceBuffers buffers = allocateDeviceBuffers(events);
  CHECK(cudaMemcpy(
      buffers.deltas, stream.deltas.data(), events * sizeof(int), cudaMemcpyHostToDevice));

  PeakLoad peak = {};
  auto run = [&] {
    peak = gpuRentalsOutAndPeak(buffers, events);
  };
  for (int warmup = 0; warmup < WARMUP_RUNS; ++warmup) {
    run();
  }
  CHECK(cudaMemset(buffers.out, 0, events * sizeof(int)));
  CHECK(cudaDeviceSynchronize());
  const double gpuMs = averageMs(TIMED_RUNS, run);
  CHECK(cudaGetLastError());

  std::vector<int> out(events);
  CHECK(cudaMemcpy(out.data(), buffers.out, events * sizeof(int), cudaMemcpyDeviceToHost));

  const bool scanPass = (out == expectedOut);
  const bool peakPass =
      (peak.rentalsOut == expectedPeak.rentalsOut) && (peak.firstEvent == expectedPeak.firstEvent);
  const bool everyBikeDocked = (out[events - 1] == 0);

  std::printf("  %-34s %10s %8s  %s\n", "step", "time [ms]", "vs CPU", "check");
  std::printf("  %-34s %10.3f %8s  %s\n", "CPU running sum + max (reference)", cpuMs, "1.00x", "-");
  std::printf("  %-34s %10.3f %7.2fx  scan %s, peak %s, ends at 0 %s\n",
              "GPU scan + max + first index",
              gpuMs,
              cpuMs / gpuMs,
              verdict(scanPass),
              verdict(peakPass),
              verdict(everyBikeDocked));

  printPeak(trips, stream, peak);
  printHourlyCurve(stream, out, expectedPeak.rentalsOut);
  printEmulationNote();

  const bool allPass = scanPass && peakPass && everyBikeDocked;
  std::printf("\n%s\n", allPass ? "ALL PASS" : "SOME CHECKS FAILED");

  freeDeviceBuffers(buffers);
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_rentals_in_progress.cu -o bikeshare_rentals_in_progress && ./bikeshare_rentals_in_progress

## `scan.cu`

every block scan (Kogge-Stone, double-buffered, Brent-Kung, coarsened), the hierarchical scan of any length, the single-pass domino scan with a ticket, exclusive from inclusive, work/step table, verification and timing

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile scan.cu
#include <algorithm>
#include <climits>
#include <cstdio>
#include <cstdlib>
#include <random>
#include <vector>

#include "cuda_compat.h"

constexpr int KS_SECTION = 256;
constexpr int KS_THREADS = KS_SECTION;

constexpr int BK_THREADS = 256;
constexpr int BK_ELEMENTS_PER_THREAD = 2;
constexpr int BK_SECTION = BK_ELEMENTS_PER_THREAD * BK_THREADS;

constexpr int CO_THREADS = 256;
constexpr int CO_COARSEN = 8;
constexpr int CO_SECTION = CO_THREADS * CO_COARSEN;

constexpr int OFFSET_THREADS = 256;
constexpr int SHIFT_THREADS = 256;
constexpr int FLAG_READY = 1;

constexpr int DEFAULT_ELEMENTS = BY_BACKEND(16000003, 100003);
constexpr int TIMED_RUNS = BY_BACKEND(5, 1);
constexpr int WARMUP_RUNS = BY_BACKEND(1, 0);
constexpr unsigned INPUT_SEED = 2025;
constexpr int MAX_DIGIT = 9;

static_assert(static_cast<long long>(DEFAULT_ELEMENTS) * MAX_DIGIT < INT_MAX,
              "the scan of the default input must fit in an int");

__global__ void kogge_stone_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[KS_SECTION];
  const int t = threadIdx.x;
  const int i = blockIdx.x * KS_SECTION + t;
  XY[t] = (i < n) ? X[i] : 0;

  for (int stride = 1; stride < KS_SECTION; stride *= 2) {
    __syncthreads();
    int temp = XY[t];
    if (t >= stride) {
      temp += XY[t - stride];
    }
    __syncthreads();
    XY[t] = temp;
  }

  if (i < n) {
    Y[i] = XY[t];
  }
  if (t == KS_SECTION - 1) {
    sums[blockIdx.x] = XY[t];
  }
}

__global__ void kogge_stone_double_buffer_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int buf[2][KS_SECTION];
  const int t = threadIdx.x;
  const int i = blockIdx.x * KS_SECTION + t;
  int src = 0;
  buf[src][t] = (i < n) ? X[i] : 0;

  for (int stride = 1; stride < KS_SECTION; stride *= 2) {
    __syncthreads();
    int v = buf[src][t];
    if (t >= stride) {
      v += buf[src][t - stride];
    }
    const int dst = 1 - src;
    buf[dst][t] = v;
    src = dst;
  }

  if (i < n) {
    Y[i] = buf[src][t];
  }
  if (t == KS_SECTION - 1) {
    sums[blockIdx.x] = buf[src][t];
  }
}

__global__ void brent_kung_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[BK_SECTION];
  const int t = threadIdx.x;
  const int base = blockIdx.x * BK_SECTION;
  XY[t] = (base + t < n) ? X[base + t] : 0;
  XY[t + BK_THREADS] = (base + t + BK_THREADS < n) ? X[base + t + BK_THREADS] : 0;

  for (int stride = 1; stride <= BK_THREADS; stride *= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index < BK_SECTION) {
      XY[index] += XY[index - stride];
    }
  }

  for (int stride = BK_SECTION / 4; stride > 0; stride /= 2) {
    __syncthreads();
    const int index = (t + 1) * 2 * stride - 1;
    if (index + stride < BK_SECTION) {
      XY[index + stride] += XY[index];
    }
  }

  __syncthreads();
  if (base + t < n) {
    Y[base + t] = XY[t];
  }
  if (base + t + BK_THREADS < n) {
    Y[base + t + BK_THREADS] = XY[t + BK_THREADS];
  }
  if (t == 0) {
    sums[blockIdx.x] = XY[BK_SECTION - 1];
  }
}

__device__ void load_section_coalesced(const int* X, int n, int base, int* XY) {
  for (int k = 0; k < CO_COARSEN; ++k) {
    const int j = k * CO_THREADS + threadIdx.x;
    XY[j] = (base + j < n) ? X[base + j] : 0;
  }
}

__device__ void store_section_coalesced(const int* XY, int offset, int n, int base, int* Y) {
  for (int k = 0; k < CO_COARSEN; ++k) {
    const int j = k * CO_THREADS + threadIdx.x;
    if (base + j < n) {
      Y[base + j] = XY[j] + offset;
    }
  }
}

__device__ void phase1_scan_own_subsection(int* XY, int first) {
  for (int k = 1; k < CO_COARSEN; ++k) {
    XY[first + k] += XY[first + k - 1];
  }
}

__device__ void phase2_kogge_stone_over_subsection_totals(int* part) {
  const int t = threadIdx.x;
  for (int stride = 1; stride < CO_THREADS; stride *= 2) {
    __syncthreads();
    int temp = part[t];
    if (t >= stride) {
      temp += part[t - stride];
    }
    __syncthreads();
    part[t] = temp;
  }
}

__device__ void phase3_add_total_of_subsections_to_the_left(int* XY, const int* part, int first) {
  const int t = threadIdx.x;
  if (t > 0) {
    const int add = part[t - 1];
    for (int k = 0; k < CO_COARSEN; ++k) {
      XY[first + k] += add;
    }
  }
}

__device__ void coarsened_block_scan(const int* X, int n, int base, int* XY, int* part) {
  const int t = threadIdx.x;
  const int first = t * CO_COARSEN;

  load_section_coalesced(X, n, base, XY);
  __syncthreads();

  phase1_scan_own_subsection(XY, first);
  part[t] = XY[first + CO_COARSEN - 1];

  phase2_kogge_stone_over_subsection_totals(part);
  __syncthreads();

  phase3_add_total_of_subsections_to_the_left(XY, part, first);
  __syncthreads();
}

__global__ void coarsened_scan(const int* X, int* Y, int* sums, int n) {
  __shared__ int XY[CO_SECTION];
  __shared__ int part[CO_THREADS];
  const int base = blockIdx.x * CO_SECTION;

  coarsened_block_scan(X, n, base, XY, part);

  store_section_coalesced(XY, 0, n, base, Y);
  if (threadIdx.x == 0) {
    sums[blockIdx.x] = part[CO_THREADS - 1];
  }
}

__global__ void add_block_offsets(int* Y, const int* scannedSums, int n, int section) {
  if (blockIdx.x > 0) {
    const int add = scannedSums[blockIdx.x - 1];
    for (int j = threadIdx.x; j < section; j += blockDim.x) {
      const long long idx = (long long)blockIdx.x * section + j;
      if (idx < n) {
        Y[idx] += add;
      }
    }
  }
}

__device__ int next_ticket(int* ticket) {
  return atomicAdd(ticket, 1);
}

__device__ int atomic_read(int* address) {
  return atomicAdd(address, 0);
}

__device__ void wait_until_ready(int* flag) {
  int state = atomic_read(flag);
  while (state != FLAG_READY) {
    state = atomic_read(flag);
  }
}

__device__ int wait_for_sum_of_previous_sections(int* flags, int* prefix, int bid) {
  if (bid == 0) {
    return 0;
  }
  wait_until_ready(&flags[bid]);
  return atomic_read(&prefix[bid]);
}

__device__ void publish_sum_of_sections_up_to(int bid, int sum, int* flags, int* prefix) {
  atomicExch(&prefix[bid + 1], sum);
  __threadfence();
  atomicExch(&flags[bid + 1], FLAG_READY);
}

__global__ void domino_scan(const int* X, int* Y, int* flags, int* prefix, int* ticket, int n) {
  __shared__ int XY[CO_SECTION];
  __shared__ int part[CO_THREADS];
  __shared__ int bid_s;
  __shared__ int before_s;

  if (threadIdx.x == 0) {
    bid_s = next_ticket(ticket);
  }
  __syncthreads();
  const int bid = bid_s;
  const int base = bid * CO_SECTION;

  coarsened_block_scan(X, n, base, XY, part);

  if (threadIdx.x == 0) {
    const int before = wait_for_sum_of_previous_sections(flags, prefix, bid);
    publish_sum_of_sections_up_to(bid, before + part[CO_THREADS - 1], flags, prefix);
    before_s = before;
  }
  __syncthreads();

  store_section_coalesced(XY, before_s, n, base, Y);
}

__global__ void inclusive_to_exclusive(const int* Y, int* E, int n) {
  const int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    E[i] = (i == 0) ? 0 : Y[i - 1];
  }
}

using BlockScanKernel = void (*)(const int*, int*, int*, int);

enum class ScanStrategy { Hierarchical, SinglePass };

struct ScanVariant {
  const char* name;
  ScanStrategy strategy;
  BlockScanKernel blockScan;
  int section;
  int threads;
};

const ScanVariant SCAN_VARIANTS[] = {
    {"hierarchical + Kogge-Stone",
     ScanStrategy::Hierarchical,
     kogge_stone_scan,
     KS_SECTION,
     KS_THREADS},
    {"hierarchical + Kogge-Stone 2-buffer",
     ScanStrategy::Hierarchical,
     kogge_stone_double_buffer_scan,
     KS_SECTION,
     KS_THREADS},
    {"hierarchical + Brent-Kung",
     ScanStrategy::Hierarchical,
     brent_kung_scan,
     BK_SECTION,
     BK_THREADS},
    {"hierarchical + coarsened 3-phase",
     ScanStrategy::Hierarchical,
     coarsened_scan,
     CO_SECTION,
     CO_THREADS},
    {"single-pass domino (coarsened)", ScanStrategy::SinglePass, nullptr, CO_SECTION, CO_THREADS},
};

struct ScanBuffers {
  int* input;
  int* output;
  int* scratch;
  int* flags;
  int* prefix;
  int* ticket;
};

int blocksFor(int n, int section) {
  return static_cast<int>(cdiv(n, section));
}

int* allocateDeviceInts(int count) {
  int* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(int)));
  return pointer;
}

void copyToDevice(int* destination, const std::vector<int>& source) {
  CHECK(
      cudaMemcpy(destination, source.data(), source.size() * sizeof(int), cudaMemcpyHostToDevice));
}

void copyToHost(std::vector<int>& destination, const int* source) {
  CHECK(cudaMemcpy(
      destination.data(), source, destination.size() * sizeof(int), cudaMemcpyDeviceToHost));
}

int scratchIntsForHierarchicalScan(int n, int section) {
  const int blocks = blocksFor(n, section);
  if (blocks == 1) {
    return 1;
  }
  return 2 * blocks + scratchIntsForHierarchicalScan(blocks, section);
}

void scanHierarchical(BlockScanKernel blockScan,
                      int section,
                      int threads,
                      const int* d_in,
                      int* d_out,
                      int n,
                      int* scratch) {
  const int blocks = blocksFor(n, section);
  int* blockSums = scratch;
  launch(blockScan, dim3(blocks), dim3(threads), d_in, d_out, blockSums, n);

  if (blocks > 1) {
    int* scannedBlockSums = scratch + blocks;
    int* scratchForNextLevel = scratch + 2 * blocks;
    scanHierarchical(
        blockScan, section, threads, blockSums, scannedBlockSums, blocks, scratchForNextLevel);
    launch(
        add_block_offsets, dim3(blocks), dim3(OFFSET_THREADS), d_out, scannedBlockSums, n, section);
  }
}

void resetDominoChain(const ScanBuffers& buffers, int sections) {
  const int chainEntries = sections + 1;
  CHECK(cudaMemset(buffers.flags, 0, chainEntries * sizeof(int)));
  CHECK(cudaMemset(buffers.prefix, 0, chainEntries * sizeof(int)));
  CHECK(cudaMemset(buffers.ticket, 0, sizeof(int)));
}

void scanSinglePass(const ScanBuffers& buffers, int n) {
  const int sections = blocksFor(n, CO_SECTION);
  resetDominoChain(buffers, sections);
  launch(domino_scan,
         dim3(sections),
         dim3(CO_THREADS),
         buffers.input,
         buffers.output,
         buffers.flags,
         buffers.prefix,
         buffers.ticket,
         n);
}

void runScanVariant(const ScanVariant& variant, const ScanBuffers& buffers, int n) {
  switch (variant.strategy) {
    case ScanStrategy::Hierarchical:
      scanHierarchical(variant.blockScan,
                       variant.section,
                       variant.threads,
                       buffers.input,
                       buffers.output,
                       n,
                       buffers.scratch);
      break;
    case ScanStrategy::SinglePass:
      scanSinglePass(buffers, n);
      break;
  }
}

void shiftToExclusive(const int* d_inclusive, int* d_exclusive, int n) {
  launch(inclusive_to_exclusive,
         dim3(blocksFor(n, SHIFT_THREADS)),
         dim3(SHIFT_THREADS),
         d_inclusive,
         d_exclusive,
         n);
}

void cpuInclusiveScan(const std::vector<int>& x, std::vector<int>& y) {
  int run = 0;
  for (size_t i = 0; i < x.size(); ++i) {
    run += x[i];
    y[i] = run;
  }
}

bool isExclusiveScanOf(const std::vector<int>& exclusive, const std::vector<int>& inclusive) {
  if (exclusive[0] != 0) {
    return false;
  }
  for (size_t i = 1; i < exclusive.size(); ++i) {
    if (exclusive[i] != inclusive[i - 1]) {
      return false;
    }
  }
  return true;
}

int firstMismatch(const std::vector<int>& actual, const std::vector<int>& expected) {
  return static_cast<int>(std::mismatch(actual.begin(), actual.end(), expected.begin()).first -
                          actual.begin());
}

template <class Work>
double averageMs(int runs, Work work) {
  Timer timer;
  for (int run = 0; run < runs; ++run) {
    work();
  }
  return timer.ms() / runs;
}

const char* verdict(bool pass) {
  return pass ? "PASS" : "FAIL";
}

void printRow(const char* label, const std::vector<int>& values) {
  std::printf("  %-10s", label);
  for (int value : values) {
    std::printf("%4d", value);
  }
  std::printf("\n");
}

bool runTinyExample() {
  const std::vector<int> input = {3, 1, 7, 0, 4, 1, 6, 3, 2, 5, 0, 8, 1, 1, 9, 2};
  const int m = static_cast<int>(input.size());
  std::vector<int> inclusive(m);
  std::vector<int> exclusive(m);

  int* d_input = allocateDeviceInts(m);
  int* d_inclusive = allocateDeviceInts(m);
  int* d_exclusive = allocateDeviceInts(m);
  int* d_blockSum = allocateDeviceInts(1);
  copyToDevice(d_input, input);

  launch(brent_kung_scan, dim3(1), dim3(BK_THREADS), d_input, d_inclusive, d_blockSum, m);
  shiftToExclusive(d_inclusive, d_exclusive, m);
  CHECK(cudaGetLastError());
  copyToHost(inclusive, d_inclusive);
  copyToHost(exclusive, d_exclusive);

  std::printf("Tiny example (Brent-Kung, one block, %d elements padded with zeros to %d):\n",
              m,
              BK_SECTION);
  printRow("input", input);
  printRow("inclusive", inclusive);
  printRow("exclusive", exclusive);

  std::vector<int> reference(m);
  cpuInclusiveScan(input, reference);
  const bool pass = (reference == inclusive) && isExclusiveScanOf(exclusive, reference);
  std::printf("  check: %s\n\n", verdict(pass));

  CHECK(cudaFree(d_input));
  CHECK(cudaFree(d_inclusive));
  CHECK(cudaFree(d_exclusive));
  CHECK(cudaFree(d_blockSum));
  return pass;
}

int log2Floor(int value) {
  int exponent = 0;
  while ((1 << (exponent + 1)) <= value) {
    ++exponent;
  }
  return exponent;
}

int sequentialAdds(int N) {
  return N - 1;
}

int koggeStoneAdds(int N) {
  return N * log2Floor(N) - (N - 1);
}

int koggeStoneSteps(int N) {
  return log2Floor(N);
}

int brentKungAdds(int N) {
  return 2 * N - 2 - log2Floor(N);
}

int brentKungSteps(int N) {
  return 2 * log2Floor(N) - 1;
}

void printWorkAndStepsTable() {
  const int sectionSizes[] = {16, 256, 512, 2048};
  std::printf("Work (additions) and steps to scan one section of N elements:\n");
  std::printf("  %6s | %10s %6s | %10s %6s | %10s %6s\n",
              "N",
              "seq adds",
              "steps",
              "KS adds",
              "steps",
              "BK adds",
              "steps");
  for (int N : sectionSizes) {
    std::printf("  %6d | %10d %6d | %10d %6d | %10d %6d\n",
                N,
                sequentialAdds(N),
                sequentialAdds(N),
                koggeStoneAdds(N),
                koggeStoneSteps(N),
                brentKungAdds(N),
                brentKungSteps(N));
  }
  std::printf("\n");
}

std::vector<int> randomDigits(int n) {
  std::mt19937 rng(INPUT_SEED);
  std::uniform_int_distribution<int> digit(0, MAX_DIGIT);
  std::vector<int> values(n);
  for (int& value : values) {
    value = digit(rng);
  }
  return values;
}

int scratchIntsForEveryVariant(int n) {
  int largest = 0;
  for (const ScanVariant& variant : SCAN_VARIANTS) {
    largest = imax(largest, scratchIntsForHierarchicalScan(n, variant.section));
  }
  return largest;
}

ScanBuffers allocateScanBuffers(int n) {
  const int chainEntries = blocksFor(n, CO_SECTION) + 1;
  ScanBuffers buffers;
  buffers.input = allocateDeviceInts(n);
  buffers.output = allocateDeviceInts(n);
  buffers.scratch = allocateDeviceInts(scratchIntsForEveryVariant(n));
  buffers.flags = allocateDeviceInts(chainEntries);
  buffers.prefix = allocateDeviceInts(chainEntries);
  buffers.ticket = allocateDeviceInts(1);
  return buffers;
}

void freeScanBuffers(const ScanBuffers& buffers) {
  CHECK(cudaFree(buffers.input));
  CHECK(cudaFree(buffers.output));
  CHECK(cudaFree(buffers.scratch));
  CHECK(cudaFree(buffers.flags));
  CHECK(cudaFree(buffers.prefix));
  CHECK(cudaFree(buffers.ticket));
}

void clearOutputSoNoStaleResultCanPass(const ScanBuffers& buffers, int n) {
  CHECK(cudaMemset(buffers.output, 0, n * sizeof(int)));
  CHECK(cudaDeviceSynchronize());
}

void printVariantTableHeader(int n, double cpuMs) {
  std::printf("Inclusive scan of n = %d ints (values 0..%d), %d timed run(s) each:\n",
              n,
              MAX_DIGIT,
              TIMED_RUNS);
  std::printf("  %-37s %8s %8s %10s %8s  %s\n",
              "variant",
              "section",
              "blocks",
              "time [ms]",
              "vs CPU",
              "check");
  std::printf("  %-37s %8s %8s %10.3f %8s  %s\n",
              "CPU sequential loop (reference)",
              "-",
              "-",
              cpuMs,
              "1.00x",
              "-");
}

bool measureAndCheckVariant(const ScanVariant& variant,
                            const ScanBuffers& buffers,
                            const std::vector<int>& reference,
                            double cpuMs) {
  const int n = static_cast<int>(reference.size());
  auto run = [&] {
    runScanVariant(variant, buffers, n);
  };

  for (int warmup = 0; warmup < WARMUP_RUNS; ++warmup) {
    run();
  }
  clearOutputSoNoStaleResultCanPass(buffers, n);
  const double ms = averageMs(TIMED_RUNS, run);
  CHECK(cudaGetLastError());

  std::vector<int> output(n);
  copyToHost(output, buffers.output);
  const int mismatch = firstMismatch(output, reference);
  const bool pass = (mismatch == n);

  std::printf("  %-37s %8d %8d %10.3f %7.2fx  %s",
              variant.name,
              variant.section,
              blocksFor(n, variant.section),
              ms,
              cpuMs / ms,
              verdict(pass));
  if (!pass) {
    std::printf(" (Y[%d] = %d, expected %d)", mismatch, output[mismatch], reference[mismatch]);
  }
  std::printf("\n");
  return pass;
}

bool checkExclusiveScan(const ScanBuffers& buffers, const std::vector<int>& reference) {
  const int n = static_cast<int>(reference.size());
  scanHierarchical(
      coarsened_scan, CO_SECTION, CO_THREADS, buffers.input, buffers.output, n, buffers.scratch);

  int* d_exclusive = allocateDeviceInts(n);
  shiftToExclusive(buffers.output, d_exclusive, n);
  CHECK(cudaGetLastError());
  std::vector<int> exclusive(n);
  copyToHost(exclusive, d_exclusive);
  CHECK(cudaFree(d_exclusive));

  const bool pass = isExclusiveScanOf(exclusive, reference);
  std::printf("  %-37s %8s %8s %10s %8s  %s\n",
              "exclusive scan (shifted inclusive)",
              "-",
              "-",
              "-",
              "-",
              verdict(pass));
  return pass;
}

void printEmulationNote() {
#ifndef __CUDACC__
  std::printf(
      "\nNote: CPU emulation times measure thread barriers, not a GPU; use them only to see that\n"
      "every variant runs. Run on a GPU (Colab) for meaningful timings.\n");
#endif
}

int elementCount(int argc, char** argv) {
  return (argc > 1) ? std::atoi(argv[1]) : DEFAULT_ELEMENTS;
}

int main(int argc, char** argv) {
  const int n = elementCount(argc, argv);
  if (n < 1) {
    std::fprintf(stderr, "n must be >= 1\n");
    return 1;
  }
  std::printf("Parallel scan (chapter 11)  backend: %s\n\n", CUDA_BACKEND);

  bool allPass = runTinyExample();
  printWorkAndStepsTable();

  const std::vector<int> input = randomDigits(n);
  std::vector<int> reference(n);
  const double cpuMs = averageMs(TIMED_RUNS, [&] {
    cpuInclusiveScan(input, reference);
  });

  const ScanBuffers buffers = allocateScanBuffers(n);
  copyToDevice(buffers.input, input);

  printVariantTableHeader(n, cpuMs);
  for (const ScanVariant& variant : SCAN_VARIANTS) {
    const bool pass = measureAndCheckVariant(variant, buffers, reference, cpuMs);
    allPass = allPass && pass;
  }
  const bool exclusivePass = checkExclusiveScan(buffers, reference);
  allPass = allPass && exclusivePass;

  printEmulationNote();
  std::printf("\nTotal of all elements: %d\n", reference[n - 1]);
  std::printf("%s\n", allPass ? "ALL PASS" : "SOME CHECKS FAILED");

  freeScanBuffers(buffers);
  return allPass ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. scan.cu -o scan && ./scan

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/09_scan/README.md`.

**Next:** open `ec2/10_filter/colab_1_step_by_step.ipynb`.